In [2]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## Breast Cancer Dataset

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=10):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)



c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 917us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 906us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.998
DecisionTree: 0.998
NDT: 0.991


## Iris Dataset

In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_iris()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=4):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 925us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 973us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 957us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 998us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.995
DecisionTree: 1.000
NDT: 0.978


## Wine Dataset

In [4]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_wine()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=13):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 869us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 953us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.999
DecisionTree: 0.997
NDT: 0.998


## California Housing Dataset

In [5]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=8):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]
    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 917us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 915us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 860us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.999
DecisionTree: 0.999
NDT: 0.974


## Diabetes Dataset

In [6]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=10):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]
    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 871us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 963us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 900us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.999
DecisionTree: 0.999
NDT: 0.998


## Digits Dataset

In [4]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_digits()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=13):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 952us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 985us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.972
DecisionTree: 0.954
NDT: 0.811


## Covtype Dataset

In [5]:
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_covtype()
X = data.data[:2000]
y = data.target[:2000]
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=13):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.990
DecisionTree: 0.997
NDT: 0.956


## Ames Housing Dataset

In [7]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_openml(name='house_prices', as_frame=True)
X = data.data.select_dtypes(include=[np.number]).dropna(axis=1)
y = data.target.astype(float)
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=8):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]
    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test.iloc[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")

c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomFor

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 906us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 914us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.998
DecisionTree: 0.999
NDT: 0.991
